To build and run Boltz2 you need an API key from NVIDIA. If you do not have one, the script 'check_boltz2_api_key.sh' should help you have a valid, secure and personal API key saved at ~/ngc/ngc_api_key.boltz2

In [ ]:
API_KEY= None  # papermill overrides this when run non-interactively

In [ ]:
import os
if not API_KEY:
    with open(os.path.expanduser("~/.ngc/ngc_api_key.genmol")) as f:
        API_KEY = f.read().strip()
os.environ["NGC_API_KEY"] = API_KEY
os.environ["CACHEDIR"] = "/nesi/nobackup/uoa04517/cache"
os.environ["LOCAL_NVS_CACHE"] = "/nesi/nobackup/uoa04517/cache"

In [ ]:
#Start Boltz2 API server
import subprocess, time

log_path = "server.log"
proc = subprocess.Popen(
    "apptainer run --env NVIDIA_VISIBLE_DEVICES=0 --env NGC_API_KEY=$NGC_API_KEY "
    "--bind $LOCAL_NVS_CACHE:/home/nvs/.cache --bind $CACHEDIR:/tmp --writable-tmpfs --nv "
      f"boltz2.sif /usr/local/bin/start_server > {log_path} 2>&1",
      shell=True,
)

time.sleep(2)  # assume server.log exists after a sleep
f = open(log_path)

while True:
    line = f.readline()
    if line:
        if "Uvicorn running" in line:
            break
    elif proc.poll() is not None:
        raise RuntimeError(f"Server exited early (code {proc.returncode}, check {log_path})")
    else:
        time.sleep(1)

print("Boltz2 server ready.")

In [ ]:
import json
import requests
from typing import Dict, Any

base_url = f"http://localhost:8000"
SEQUENCE = "MTEYKLVVVGACGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVID"

def query_boltz2_nim(input_data: Dict[str, Any], base_url: str = base_url) -> Dict[str, Any]:
    url = f"{base_url}/biology/mit/boltz2/predict"
    headers = {"Content-Type": "application/json"}
    
    response = requests.post(url, headers=headers, json=input_data)
    response.raise_for_status()
    return response.json()

if __name__ == "__main__":
    example_input = {
        "polymers": [
            {
                "id": "A",
                "molecule_type": "protein",
                "sequence": SEQUENCE
            }
        ]
    }
    
    try:
        result = query_boltz2_nim(example_input)
        print("Prediction result:")
        print(json.dumps(result, indent=2))
    except Exception as e:
        print(f"Failed to get prediction: {e}")
